In [1]:
# ============================================
# Orthomosaic → Tiles for ML labeling/training
# Requires: rasterio, numpy
# Optional: matplotlib (preview), pillow (PNG)
# Install (if needed):
#   pip install rasterio numpy matplotlib pillow
# ============================================


In [2]:
!pip install rasterio numpy matplotlib pillow


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip


In [3]:
# 1 - Imports & configuration

from pathlib import Path
import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.windows import transform as window_transform

# ---- USER SETTINGS ----
ORTHO_PATH = Path("./data_set/orthomosaic/2022_MetroVAN_7_5cm.tif")
OUT_DIR    = Path("./output/tiles_out")                # output folder

TILE_SIZE  = 512          # 256 or 512 are common
OVERLAP    = 128          # 0, 64, 128... (overlap pixels). For no overlap set 0.
SKIP_EMPTY_THRESHOLD = 0.90  # skip tiles if >=90% nodata/black

# Output format:
# - "geotiff": preserves georeferencing (recommended for later GIS work)
# - "png": easy for labeling tools, but no georeferencing (we write a JSON sidecar)
OUTPUT_FORMAT = "png"     # "geotiff" or "png"

# If your ortho has many bands, choose RGB:
BANDS = (1, 2, 3)         # change if needed

OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "images").mkdir(exist_ok=True)
(OUT_DIR / "meta").mkdir(exist_ok=True)

print("Config OK")


Config OK


In [4]:
# 2 — Helper functions

def compute_grid(width, height, tile_size, overlap):
    stride = tile_size - overlap
    if stride <= 0:
        raise ValueError("OVERLAP must be < TILE_SIZE")

    xs = list(range(0, max(1, width - tile_size + 1), stride))
    ys = list(range(0, max(1, height - tile_size + 1), stride))

    # ensure right/bottom edges are covered
    if xs[-1] != width - tile_size:
        xs.append(max(0, width - tile_size))
    if ys[-1] != height - tile_size:
        ys.append(max(0, height - tile_size))

    return xs, ys, stride


def empty_fraction(arr, nodata=None):
    """
    arr: (bands, H, W)
    empty pixels:
      - if nodata exists: all bands == nodata
      - else: near-black (all bands <= 2)
    """
    if nodata is not None:
        mask = np.all(arr == nodata, axis=0)
    else:
        mask = np.all(arr <= 2, axis=0)
    return float(mask.mean())


def to_uint8_rgb(arr):
    """
    Convert (bands,H,W) to uint8 for PNG.
    If already uint8, return.
    If uint16/float: percentile stretch per band.
    """
    if arr.dtype == np.uint8:
        return arr

    out = np.zeros_like(arr, dtype=np.uint8)
    for b in range(arr.shape[0]):
        band = arr[b].astype(np.float32)
        lo = np.percentile(band, 1)
        hi = np.percentile(band, 99)
        if hi <= lo:
            hi = lo + 1.0
        band = (band - lo) / (hi - lo)
        band = np.clip(band, 0, 1)
        out[b] = (band * 255).round().astype(np.uint8)
    return out


In [5]:
# 3 — Tile the orthomosaic

import json

try:
    from PIL import Image
except ImportError:
    Image = None

tiles_index = []  # will store metadata for all tiles we saved

with rasterio.open(ORTHO_PATH) as src:
    width, height = src.width, src.height
    nodata = src.nodata
    crs = src.crs
    base_transform = src.transform

    xs, ys, stride = compute_grid(width, height, TILE_SIZE, OVERLAP)

    print("Ortho size (px):", width, "x", height)
    print("Tile size:", TILE_SIZE, "Overlap:", OVERLAP, "Stride:", stride)
    print("CRS:", crs)
    print("Nodata:", nodata)

    kept = 0
    total = 0

    for y in ys:
        for x in xs:
            total += 1
            win = Window(col_off=x, row_off=y, width=TILE_SIZE, height=TILE_SIZE)

            # Read selected bands
            arr = src.read(indexes=BANDS, window=win)

            # Skip partial tiles (edges) if any
            if arr.shape[1] != TILE_SIZE or arr.shape[2] != TILE_SIZE:
                continue

            if empty_fraction(arr, nodata=nodata) >= SKIP_EMPTY_THRESHOLD:
                continue

            kept += 1
            tile_name = f"tile_x{x}_y{y}_s{TILE_SIZE}"

            # Per-tile transform (georeferencing)
            tform = window_transform(win, base_transform)

            if OUTPUT_FORMAT.lower() == "geotiff":
                out_path = OUT_DIR / "images" / f"{tile_name}.tif"
                profile = src.profile.copy()
                profile.update(
                    height=TILE_SIZE,
                    width=TILE_SIZE,
                    transform=tform,
                    count=len(BANDS),
                    compress="deflate",
                    tiled=False,
                )
                with rasterio.open(out_path, "w", **profile) as dst:
                    dst.write(arr)

            elif OUTPUT_FORMAT.lower() == "png":
                if Image is None:
                    raise ImportError("Install pillow for PNG output: pip install pillow")

                arr8 = to_uint8_rgb(arr)
                # Convert CHW -> HWC for PIL
                hwc = np.transpose(arr8, (1, 2, 0))
                out_path = OUT_DIR / "images" / f"{tile_name}.png"
                Image.fromarray(hwc).save(out_path)

            else:
                raise ValueError("OUTPUT_FORMAT must be 'geotiff' or 'png'")

            # Save metadata (so you can always map tile back to ortho)
            meta = {
                "tile_name": tile_name,
                "image_file": str(out_path.name),
                "x_off_px": int(x),
                "y_off_px": int(y),
                "tile_size_px": int(TILE_SIZE),
                "overlap_px": int(OVERLAP),
                "stride_px": int(stride),
                "bands": list(BANDS),
                "crs": str(crs),
                # Affine transform: [a, b, c, d, e, f] corresponds to:
                # x = a*col + b*row + c
                # y = d*col + e*row + f
                "transform": [tform.a, tform.b, tform.c, tform.d, tform.e, tform.f],
            }

            (OUT_DIR / "meta" / f"{tile_name}.json").write_text(json.dumps(meta, indent=2))
            tiles_index.append(meta)

print(f"Total windows considered: {total}")
print(f"Tiles saved: {kept}")
print("Tiles folder:", OUT_DIR / "images")
print("Metadata folder:", OUT_DIR / "meta")


RasterioIOError: 2022_MetroVAN_7_5cm.tif: TIFFReadDirectory:Failed to read directory at offset 6588508772